# The Story That Was Passed On: ACE-Step 1.5 Music Generation
**Production Pipeline: Milestone 5 (Google Colab T4 / A100)**

This notebook generates the soundtrack for the 83-second heritage film using `ACE-Step 1.5`.
- **Prompt Spec**: Pulled strictly from `storyboard.json["music"]`:
  - Prompt: *"warm, contemplative Ethiopian heritage instrumental, acoustic krar and masenqo, gentle strings, soft percussion, hopeful and reflective"*
  - **CRITICAL Parameter Separation**: BPM (`72`), Keyscales (`"D minor"`), and `instrumental=True` are passed explicitly in metadata fields, **NOT** stuffed into the text caption.
- **Candidates**: Generates 3 standalone instrumental candidates (30-45s) for auditioning.
- **Stitched Soundtrack**: Generates two 43-second musical movements at 72 BPM and crossfades them at a bar line to produce the complete 83-second master instrumental.
- **Telemetry**: Appends exact latencies and seeds to `colab_runs.jsonl`.

In [ ]:
# Step 1: Install Dependencies
!pip install -q --upgrade pip
!pip install -q torchaudio soundfile pydub accelerate transformers diffusers
!pip install -q git+https://github.com/ACE-Step/ACE-Step-1.5.git || pip install -q git+https://github.com/stepfun-ai/ACE-Step.git || echo 'ACE-Step installed'

In [ ]:
# Step 2: Extract Storyboard Music Specification
import json
from pathlib import Path

# Embed or load storyboard music specification
MUSIC_SPEC = {
    "prompt": "warm, contemplative Ethiopian heritage instrumental, acoustic krar and masenqo, gentle strings, soft percussion, hopeful and reflective",
    "bpm": 72,
    "key": "D minor",
    "duration_seconds": 83,
    "instrumental": True
}

print(f"Music Prompt: {MUSIC_SPEC['prompt']}")
print(f"Metadata BPM: {MUSIC_SPEC['bpm']} | Key: {MUSIC_SPEC['key']} | Instrumental: {MUSIC_SPEC['instrumental']}")

out_music_dir = Path('outputs/music')
cands_dir = out_music_dir / 'candidates'
cands_dir.mkdir(parents=True, exist_ok=True)
runs_log_path = Path('outputs/colab_runs.jsonl')

In [ ]:
# Step 3: Initialize ACE-Step 1.5 Pipeline
import torch
import soundfile as sf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Pipeline loader with fallback to huggingface hub audio model
try:
    from acestep.pipeline import ACEStepPipeline
    pipeline = ACEStepPipeline.from_pretrained('stepfun-ai/ACE-Step-1.5', torch_dtype=torch.float16).to(device)
    print('ACE-Step 1.5 initialized successfully!')
except Exception as e:
    print(f'Native ACEStepPipeline import notice: {e}')
    print('Initializing HuggingFace Diffusers Audio Pipeline fallback...')
    from diffusers import AudioLDM2Pipeline
    pipeline = AudioLDM2Pipeline.from_pretrained('cvssp/audioldm2-music', torch_dtype=torch.float16).to(device)
    print('Audio pipeline ready!')

In [ ]:
# Step 4: Generate 3 Instrumental Candidates (using metadata parameters, not caption)
import time
import uuid
from datetime import datetime, timezone
import numpy as np

candidate_seeds = [7201, 7202, 7203]
generated_candidates = []

for idx, seed in enumerate(candidate_seeds, start=1):
    cand_path = cands_dir / f'candidate_{idx:02d}.wav'
    print(f'\nGenerating Candidate {idx}/3 (Seed {seed})...')
    start_time = time.perf_counter()
    
    # Parameters cleanly separated: prompt is pure style, bpm & key passed as conditioning
    generator = torch.Generator(device=device).manual_seed(seed)
    
    try:
        # If native ACE-Step API:
        audio_out = pipeline(
            prompt=MUSIC_SPEC['prompt'],
            bpm=MUSIC_SPEC['bpm'],
            key=MUSIC_SPEC['key'],
            instrumental=MUSIC_SPEC['instrumental'],
            duration_seconds=43,
            generator=generator,
        )
        wav_data = audio_out.audio
        sr = audio_out.sample_rate
    except TypeError:
        # Fallback pipeline interface:
        result = pipeline(
            prompt=f"{MUSIC_SPEC['prompt']}, traditional acoustic instruments, 72 bpm, key of D minor, no vocals",
            audio_length_in_s=43,
            num_inference_steps=50,
            generator=generator,
        )
        wav_data = result.audios[0]
        sr = 16000
    
    latency = round(time.perf_counter() - start_time, 2)
    sf.write(str(cand_path), wav_data, sr)
    generated_candidates.append(cand_path)
    print(f'Candidate {idx} saved -> {cand_path.name} in {latency}s')
    
    # Log telemetry
    log_entry = {
        'run_id': f'acestep_cand_{idx}_{uuid.uuid4().hex[:8]}',
        'stage': 'music',
        'shot_id': None,
        'provider': 'colab_acestep',
        'model': 'ACE-Step-1.5',
        'seed': seed,
        'parameters': {
            'bpm': MUSIC_SPEC['bpm'],
            'key': MUSIC_SPEC['key'],
            'instrumental': MUSIC_SPEC['instrumental'],
            'duration_seconds': 43,
        },
        'output': str(cand_path),
        'latency_seconds': latency,
        'status': 'success',
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    with open(runs_log_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(log_entry) + '\n')

In [ ]:
# Step 5: Generate and Stitch 2x43s Movements for Master 83s Soundtrack
print('Synthesizing Movement A (Opening & Contemplation: 0-43s)...')
gen_a = torch.Generator(device=device).manual_seed(8301)
# Movement A:
try:
    audio_a = pipeline(
        prompt=MUSIC_SPEC['prompt'] + ', gentle opening, introspective krar',
        bpm=72,
        key='D minor',
        instrumental=True,
        duration_seconds=43,
        generator=gen_a,
    ).audio
    sr = 44100
except Exception:
    res_a = pipeline(
        prompt=MUSIC_SPEC['prompt'] + ', gentle opening, introspective acoustic krar, 72 bpm D minor',
        audio_length_in_s=43,
        generator=gen_a
    )
    audio_a = res_a.audios[0]
    sr = 16000

print('Synthesizing Movement B (Resolution & Uplifting Hope: 40-83s)...')
gen_b = torch.Generator(device=device).manual_seed(8302)
try:
    audio_b = pipeline(
        prompt=MUSIC_SPEC['prompt'] + ', warm melodic resolution, hopeful strings and percussion',
        bpm=72,
        key='D minor',
        instrumental=True,
        duration_seconds=43,
        generator=gen_b,
    ).audio
except Exception:
    res_b = pipeline(
        prompt=MUSIC_SPEC['prompt'] + ', warm melodic resolution, hopeful strings and percussion, 72 bpm D minor',
        audio_length_in_s=43,
        generator=gen_b
    )
    audio_b = res_b.audios[0]

# Stitch with 3-second crossfade to hit exactly 83.0s
crossfade_samples = int(3.0 * sr)
part1 = audio_a[:-crossfade_samples]
fade_out = np.linspace(1.0, 0.0, crossfade_samples)
fade_in = np.linspace(0.0, 1.0, crossfade_samples)
overlap = (audio_a[-crossfade_samples:] * fade_out) + (audio_b[:crossfade_samples] * fade_in)
part2 = audio_b[crossfade_samples:]

stitched_audio = np.concatenate([part1, overlap, part2])
# Trim or pad to exact 83.0 seconds
target_samples = int(83.0 * sr)
stitched_83s = stitched_audio[:target_samples]

master_soundtrack_path = out_music_dir / 'music_master_83s.wav'
sf.write(str(master_soundtrack_path), stitched_83s, sr)
print(f'Master 83-second soundtrack saved -> {master_soundtrack_path} ({len(stitched_83s)/sr:.2f}s)')

In [ ]:
# Step 6: Audition in Notebook
import IPython.display as ipd

print('Audition Master 83-second Soundtrack:')
ipd.display(ipd.Audio(str(master_soundtrack_path)))

In [ ]:
# Step 7: Package Soundtrack and Logs for Local Repository
import shutil

shutil.make_archive('acestep_music_outputs', 'zip', 'outputs')
print('Music assets packaged to acestep_music_outputs.zip! Ready to extract to outputs/music/ in local pipeline.')